In [11]:
import pandas as pd
import joblib
import numpy as np
import gc

In [ ]:
model = joblib.load('isolation_forest_model.joblib') #目前要480
# features = np.load('usable_features.npy', allow_pickle=True).tolist() # 60版本
with open('./../columns_list.txt', 'r', encoding='utf-8') as f:
    all_columns = [line.strip() for line in f if line.strip()]
features = all_columns[1:481]
target_column = all_columns[481]

d:\Users\school\python\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator ExtraTreeRegressor from version 1.6.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
d:\Users\school\python\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator IsolationForest from version 1.6.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [13]:
tp = fp = tn = fn = 0
total_count = 0
cols_to_use = features + [target_column]

In [14]:
print(features)
print(len(features))
expected_features = model.n_features_in_
print(f"模型預期的特徵數量是: {expected_features}")


['Tcp_Listen', 'layers.tcp.tcp.flags_tree.tcp.flags.ack', 'layers.tcp.tcp.stream', 'layers.ip.ip.checksum.status', 'layers.sll.sll.hatype', 'layers.tcp.tcp.analysis.tcp.analysis.acks_frame', 'Disk_Read', 'layers.ip.ip.flags_tree.ip.flags.rb', 'layers.tcp.tcp.flags_tree.tcp.flags.syn_tree._ws.expert._ws.expert.severity', 'Cached', 'publishers_count', 'SwapFree', 'layers.tcp.tcp.flags_tree.tcp.flags.syn_tree._ws.expert._ws.expert.message', 'pgdeactivate', 'Tcp_Close', 'layers.tcp.tcp.window_size', 'Buffers', 'Active', 'layers.ip.ip.checksum', 'layers.tcp.tcp.options_tree.tcp.options.nop_tree.tcp.option_kind', 'pgactivate', 'Inactive', 'msg_type', 'Net_Received', 'layers.ssl.ssl.record.ssl.record.content_type', 'layers.icmpv6.icmpv6.type', 'layers.tcp.tcp.window_size_value', 'layers.tcp.tcp.payload', 'Net_Sent', 'pgfault', 'layers.tcp.tcp.flags_tree.tcp.flags.syn', 'Tcp_Syn', 'layers.ip.ip.version', 'subscribers_count', 'layers.tcp.tcp.analysis.tcp.analysis.initial_rtt', 'layers.tcp.tcp.o

In [15]:
file_path = './../usable_temp_bin.csv'
chunk_size = 100000

In [16]:
# 讀取 CSV 的 Header
csv_headers = pd.read_csv(file_path, nrows=0).columns.tolist()
csv_headers = csv_headers[1:]
# 找出 CSV 裡這 480 個特徵的索引順序
# 假設你的 CSV 本身就含有這些欄位
X_sample = pd.read_csv(file_path, nrows=1, usecols=features)
print("CSV 讀入後的欄位順序：")
print(X_sample.columns.tolist()[:5]) # 印出前5個比對
print(csv_headers)

CSV 讀入後的欄位順序：
['layers.sll.sll.pkttype', 'layers.sll.sll.hatype', 'layers.sll.sll.unused', 'layers.ip.ip.version', 'layers.ip.ip.dsfield_tree.ip.dsfield.ecn']
['timestamp', 'layers.frame.frame.time', 'layers.frame.frame.time_delta', 'layers.frame.frame.time_delta_displayed', 'layers.frame.frame.time_relative', 'layers.frame.frame.number', 'layers.frame.frame.len', 'layers.frame.frame.cap_len', 'layers.frame.frame.protocols', 'layers.sll.sll.pkttype', 'layers.sll.sll.hatype', 'layers.sll.sll.src.eth', 'layers.sll.sll.unused', 'layers.sll.sll.etype', 'layers.ip.ip.version', 'layers.ip.ip.hdr_len', 'layers.ip.ip.dsfield', 'layers.ip.ip.dsfield_tree.ip.dsfield.dscp', 'layers.ip.ip.dsfield_tree.ip.dsfield.ecn', 'layers.ip.ip.len', 'layers.ip.ip.id', 'layers.ip.ip.flags', 'layers.ip.ip.flags_tree.ip.flags.rb', 'layers.ip.ip.flags_tree.ip.flags.df', 'layers.ip.ip.flags_tree.ip.flags.mf', 'layers.ip.ip.flags_tree.ip.frag_offset', 'layers.ip.ip.ttl', 'layers.ip.ip.proto', 'layers.ip.ip.checksum

In [17]:
def count_csv_rows(file_path):
    count = 0
    with open(file_path, 'rb') as f:
        # 使用 1MB 的緩衝區逐塊讀取
        for line in f:
            count += 1
    return count
count_csv_rows(file_path)

131377

In [18]:
print("開始分批驗證大檔案...")

開始分批驗證大檔案...


In [19]:
total_anomalies = 0
total_count = 0
reader = pd.read_csv(
    file_path, 
    chunksize=chunk_size, 
    usecols=cols_to_use,   # 關鍵：直接跳過 Timestamp 和 Attack 欄位
    engine='c'          # 使用 C 引擎讀取較快
)

In [20]:
try:
    for i, chunk in enumerate(reader):
        X = chunk[features].apply(pd.to_numeric, errors='coerce').fillna(0).astype('float32')
        y_true = chunk[target_column].values

        preds = model.predict(X)

        y_pred = np.where(preds == -1, 1, 0)

        tp += np.sum((y_pred == 1) & (y_true == 1))
        fp += np.sum((y_pred == 1) & (y_true == 0))
        tn += np.sum((y_pred == 0) & (y_true == 0))
        fn += np.sum((y_pred == 0) & (y_true == 1))

        total_count += len(y_true)

        anomalies = np.sum(preds == -1)
        total_anomalies += anomalies
        
        if i % 5 == 0:
            current_precision = tp / (tp + fp) if (tp + fp) > 0 else 0
            current_recall = tp / (tp+fn) if (tp + fn) > 0 else 0
            print(f"TP: {tp}, FP: {fp}, TN: {tn}, FN: {fn}")
            print(f"Batch {i}: 已處理 {total_count} 筆 | 累計 Precision: {current_precision:.2%}")
            print(f"Batch {i}: 已處理 {total_count} 筆 | 累計 Precision: {current_recall:.2%}")

        del chunk, X, y_true, y_pred, preds
        gc.collect()

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

    print("\n" + "="*40)
    print(f"【ROS2 異常檢測印證結果】")
    print(f"總處理樣本: {total_count}")
    print(f"TP (正確攔截): {tp} | FP (誤報): {fp}")
    print(f"TN (正常放行): {tn} | FN (漏報): {fn}")
    print("-" * 20)
    print(f"準確率 (Precision): {precision:.4%}")
    print(f"召回率 (Recall):    {recall:.4%}")
    print(f"F1-Score:           {f1:.4%}")
    print("="*40)
except Exception as e:
    print(f"執行中斷: {e}")
    

執行中斷: X has 60 features, but IsolationForest is expecting 480 features as input.


d:\Users\school\python\Lib\site-packages\sklearn\utils\validation.py:2684: UserWarning: X has feature names, but IsolationForest was fitted without feature names
  warnings.warn(
